# Source Code

This file details the code used to perform the analysis and generate the graphs. There will be step-by-step descriptions for each cell.

# Part 1: Analysis with the latest FAFB snapshot (v783)

## 1.1 Load edge list into a directed graph

The FAFB edge list was downloaded from [FlyWire](https://codex.flywire.ai/api/download).

In [20]:
import pandas as pd
import networkx as nx
from json import load

edge_list = pd.read_csv("connections_princeton.csv")
edge_list = (
    edge_list
    .groupby(['pre_root_id', 'post_root_id', 'nt_type'], as_index=False)['syn_count']
    .sum()
)
# edge_list_no_neuropil = (
#     edge_list
#     .groupby(['pre_root_id', 'post_root_id', 'nt_type'], as_index=False)['syn_count']
#     .sum()
# )
# print(edge_list.groupby(['pre_root_id','post_root_id'])['syn_count'].sum().sum())  # true total
# print(edge_list_no_neuropil['syn_count'].sum())  # after correct collapse

# dropped = edge_list.drop(columns=['neuropil']).drop_duplicates()
# print(dropped['syn_count'].sum())          # drop_duplicates total
# print(edge_list_no_neuropil['syn_count'].sum())  # groupby.sum() total, known-correct
# print(dropped['syn_count'].sum() - edge_list_no_neuropil['syn_count'].sum())
neurons = None

with open("data/groups.json") as f:
    neurons = load(f)

G = nx.from_pandas_edgelist(edge_list, source="pre_root_id", target="post_root_id", create_using=nx.DiGraph)
IR94E = set(neurons["783"]["Ir94e"])
OVIDN = set(neurons["783"]["OviDN"])

## 1.2 Find shortest paths

This cell iterates through each Ir94e, then through each OviDN, finding the shortest path between each Ir94e to each OviDN.

`nx.algorithms.all_shortest_paths` will return multiple shortest paths if they are identical in length. For each discovered path, we add the edges to a running list called `path_edges`. We also save the interneurons to a set.

Note: this performs a shortest paths analysis across the entire connectome for 90 iterations (18 Ir94e x 5 OviDN), so this cell can run for a while.

In [21]:
INTERNEURONS = set()
path_edges = []

for ir94e in neurons["783"]["Ir94e"]:
    for ovidn in neurons["783"]["OviDN"]:
        print(f"finding pathways between {ir94e} -> {ovidn}...", end="")
        paths: Generator[list, None, None] = nx.algorithms.all_shortest_paths(G, ir94e, ovidn)
        print("done")
        for path in paths:
            if len(path) > 4: continue # only 3 hops/4 neurons or less are considered as per the original paper's methods
            path_edges.extend(zip(path, path[1:])) # zip(path, path[1:]) is a neat shorthand of generating edges from the list of nodes in the path
            INTERNEURONS.update(set(path[1:-1])) # each path starts with Ir94e and ends with OviDN, so the interneurons are the nodes in between

path_edges_df = pd.DataFrame(path_edges, columns=["pre_root_id", "post_root_id"])
print(f"interneurons discovered: {len(INTERNEURONS)}")
print(f"unique path edges discovered: {len(path_edges_df.drop_duplicates())}")

finding pathways between 720575940621375231 -> 720575940632512156...done
finding pathways between 720575940621375231 -> 720575940620625880...done
finding pathways between 720575940621375231 -> 720575940621257340...done
finding pathways between 720575940621375231 -> 720575940627921182...done
finding pathways between 720575940621375231 -> 720575940642312136...done
finding pathways between 720575940638218173 -> 720575940632512156...done
finding pathways between 720575940638218173 -> 720575940620625880...done
finding pathways between 720575940638218173 -> 720575940621257340...done
finding pathways between 720575940638218173 -> 720575940627921182...done
finding pathways between 720575940638218173 -> 720575940642312136...done
finding pathways between 720575940626016017 -> 720575940632512156...done
finding pathways between 720575940626016017 -> 720575940620625880...done
finding pathways between 720575940626016017 -> 720575940621257340...done
finding pathways between 720575940626016017 -> 7205

## 1.3 Filter main edge list

Next, we filter the main edge list to capture edges involved in the circuit with their respective weights. This is done with a simple pandas merge on the source/target columns. The results are saved to a separate file.

In [22]:
circuit_edges = edge_list.merge(
    path_edges_df.drop_duplicates(),
    on=["pre_root_id", "post_root_id"],
    how="inner",
)

print(circuit_edges.head())
print(circuit_edges.shape)
circuit_edges.to_csv("filtered_edge_list.csv", index=False)

          pre_root_id        post_root_id nt_type  syn_count
0  720575940604395436  720575940620625880     ACH          7
1  720575940604395436  720575940621257340     ACH         11
2  720575940604395436  720575940642312136     ACH          5
3  720575940604891360  720575940629013199    GABA          6
4  720575940605040300  720575940642312136    GLUT         11
(302, 4)


## 1.4 Pool connectivity from the neuron level to the group level

The above edge list shows how individual neurons are connected within the graph. However, we are interested in the connectibity between *groups* of neurons.

This requires knowing what neuron IDs fall into which groups. For the sake of recreating the figure, I pulled this information directly from the paper's supplemental information (with some IDs being updated since the paper's publishing).

In [23]:
import json
import plotly.graph_objects as go
import pandas as pd

edge_list = pd.read_csv("filtered_edge_list.csv")
with open("data/groups.json") as f:
    groups = json.load(f)["783"]

# invert groups -> id_to_group
id_to_group = {}
for grp, ids in groups.items():
    for i in ids:
        id_to_group[str(i)] = grp

def id_to_grp(i):
    return id_to_group.get(str(i), "Other")

edge_list['pre_group'] = edge_list['pre_root_id'].apply(id_to_grp)
edge_list['post_group'] = edge_list['post_root_id'].apply(id_to_grp)

edge_list.groupby(['pre_group', 'post_group'])['syn_count'].sum().reset_index().to_csv("grouped_edge_list.csv", index=False)

## 1.5 Construct the Sankey

The library used to visualize the Sankey, `plotly`, uses lists of sources, targets, and values. The sources, targets, and values only take integers, which correspond to the indices of the `labels`. `name_to_idx` maps the name of the group to its index in `labels`.

Essentially, there's some boilerplate needed to represent the data in a way that `plotly` understands and can use to build the diagram.

In [24]:
import pandas as pd
import json

aggregated = pd.read_csv("grouped_edge_list.csv")

# Build global labels
name_to_idx = {}
labels = []

source = []
target = []
value = []

for group in aggregated['pre_group'].unique():
    if group not in name_to_idx:
        name_to_idx[group] = len(labels)
        labels.append(group)

for group in aggregated['post_group'].unique():
    if group not in name_to_idx:
        name_to_idx[group] = len(labels)
        labels.append(group)

for _, row in aggregated.iterrows():
    source.append(name_to_idx[row['pre_group']])
    target.append(name_to_idx[row['post_group']])
    value.append(row['syn_count'])


name_to_color = { # Key by group name for easier identification
    "Ir94e": {"line": "purple", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T1 (L)": {"line": "yellow", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T1 (R)": {"line": "yellow", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T2 (L)": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "GNG.SLP.T2 (R)": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Earmuff": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Stanley Glu interneuron": {"line": "red", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Stanley ACh interneuron": {"line": "red", "arrow": "rgba(0, 255, 0, 0.5)"},
    "OviDN": {"line": "blue", "arrow": "rgba(0, 255, 0, 0.5)"},
}

node_colors = [name_to_color.get(label, {"line": "grey"})['line'] for label in labels]
arrow_colors = [name_to_color.get(labels[src], {"arrow": "rgba(128, 128, 128, 0.5)"})['arrow'] for src in source]

# plot Sankey
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        color=node_colors
    ),
    link=dict(
        arrowlen=15,
        source=source,
        target=target,
        value=value,
        color=arrow_colors
    )
)])
fig.update_layout(title_text="Ir94e → interneuron groups → OviDN (aggregated)", font_size=10)
fig.show()

# Part 2: Comparison against v630 data

The original paper used FAFB v630, but since then FlyWire has released v783, which includes more annotations, revised edge weights, etc. The above analysis used v783, so we want see if there are any substantial differences from the published results.

## 2.1 Download edge list from materialization v630

FlyWire doesn't offer v630 datasets for download anymore, so we will have to query the live database using `fafbseg`. The original paper limited the pathway length to three hops, so we can reconstruct that by doing the following:

* Get all immediate partners of Ir94e (hop 1: Ir94e -> 1° interneuron)
* Get the partners of the 1° interneurons (hop 2: 1° interneuron -> 2° interneurons).
* Get the partners of OviDNs (hop 3: 2° interneurons -> OviDNs). Because all edges are guaranteed to be connected to OviDNs, this minimizes the search space much more than if we were to again find the neurons downstream of the 2° interneurons and then perform a pathway analysis.

I included both pre- and post-synaptic partners when performing these searches, but it doesn't matter much since we're just looking at the forward direction of the circuit. The shortest paths analysis will use the directed edges to reach OviDNs regardless. The only advantage of forcing forward directionality is that you'd be working with less data, and the search on CAVE might be a bit faster.

In [ ]:
from fafbseg import flywire
import pandas as pd
import json

neurons = json.load(open("data/groups.json"))

first_hop: pd.DataFrame = flywire.synapses.get_connectivity(neurons["630"]["Ir94e"], filtered=True, materialization=630)
first_hop = first_hop[first_hop['weight'] >= 5]
print(f"{len(first_hop)} edges in the first hop.")

second_hop: pd.DataFrame = flywire.synapses.get_connectivity(first_hop['post'].unique().tolist(), upstream=True,filtered=True, materialization=630)
second_hop = second_hop[second_hop['weight'] >= 5]
print(f"{len(second_hop)} edges in the second hop.")

print(len(second_hop))  # raw row count — compare this number directly against the pre-fix run

intermediate_ids = set(first_hop['post'].unique())
print((second_hop['post'].isin(intermediate_ids)).sum(), "rows where intermediate is still on the POST side")

third_hop: pd.DataFrame = flywire.synapses.get_connectivity(neurons["630"]["OviDN"], downstream=True, filtered=True, materialization=630)
third_hop = third_hop[third_hop['weight'] >= 5]
print(f"{len(third_hop)} edges in the third hop.")

edge_list = pd.concat([first_hop, second_hop, third_hop], ignore_index=True).drop_duplicates()
edge_list.to_csv("630_edge_list.csv", index=False)

303 edges in the first hop.


Fetching connectivity:   0%|          | 0/3 [00:00<?, ?it/s]

5322 edges in the second hop.
5322
2645 rows where intermediate is still on the POST side
235 edges in the third hop.


## 2.2 Find the shortest paths

In [42]:
import pandas as pd
import networkx as nx

edge_list = pd.read_csv("630_edge_list.csv")
G = nx.from_pandas_edgelist(edge_list, source="pre", target="post", create_using=nx.DiGraph)

path_edges = []

for ir94e in neurons["630"]["Ir94e"]:
    for ovidn in neurons["630"]["OviDN"]:
        print(f"finding pathways between {ir94e} -> {ovidn}...", end="")
        paths: Generator[list, None, None] = nx.algorithms.all_shortest_paths(G, ir94e, ovidn)
        print("done")
        for path in paths:
            if len(path) > 4: continue # probably redundant but whatever
            path_edges.extend(zip(path, path[1:]))

path_edges_df = pd.DataFrame(path_edges, columns=["pre", "post"]).drop_duplicates()
print(f"unique path edges discovered: {len(path_edges_df.drop_duplicates())} ({len(path_edges_df)} total)")

finding pathways between 720575940621375231 -> 720575940632512156...done
finding pathways between 720575940621375231 -> 720575940640872923...done
finding pathways between 720575940621375231 -> 720575940621257340...done
finding pathways between 720575940621375231 -> 720575940613316783...done
finding pathways between 720575940621375231 -> 720575940642312136...done
finding pathways between 720575940638218173 -> 720575940632512156...done
finding pathways between 720575940638218173 -> 720575940640872923...done
finding pathways between 720575940638218173 -> 720575940621257340...done
finding pathways between 720575940638218173 -> 720575940613316783...done
finding pathways between 720575940638218173 -> 720575940642312136...done
finding pathways between 720575940626016017 -> 720575940632512156...done
finding pathways between 720575940626016017 -> 720575940640872923...done
finding pathways between 720575940626016017 -> 720575940621257340...done
finding pathways between 720575940626016017 -> 7205

## 2.3 Filter the larger edge list

In [ ]:
circuit_edges = edge_list.merge(
    path_edges_df,
    on=["pre", "post"],
    how="inner",
)
circuit_edges.to_csv("630_filtered_edge_list.csv", index=False)
print(circuit_edges.shape)  # should be close to 192, same order of magnitude as v783's 302

(192, 3)


## 2.4 Pool edges to the group level

In [44]:
import json
import plotly.graph_objects as go
import pandas as pd

edge_list = pd.read_csv("630_filtered_edge_list.csv")
with open("data/groups.json") as f:
    groups = json.load(f)["630"]

# invert groups -> id_to_group
id_to_group = {}
for grp, ids in groups.items():
    for i in ids:
        id_to_group[str(i)] = grp

def id_to_grp(i):
    return id_to_group.get(str(i), "Other")

edge_list['pre_group'] = edge_list['pre'].apply(id_to_grp)
edge_list['post_group'] = edge_list['post'].apply(id_to_grp)

edge_list.groupby(['pre_group', 'post_group'])['weight'].sum().reset_index().to_csv("630_grouped_edge_list.csv", index=False)

## 2.5 Construct the Sankey

In [45]:
import pandas as pd

aggregated = pd.read_csv("630_grouped_edge_list.csv")
# aggregated = aggregated[(aggregated['pre_group'] != "Other") & (aggregated['post_group'] != "Other")]

# Build global labels
name_to_idx = {}
labels = []

source = []
target = []
value = []

for group in aggregated['pre_group'].unique():
    if group not in name_to_idx:
        name_to_idx[group] = len(labels)
        labels.append(group)

for group in aggregated['post_group'].unique():
    if group not in name_to_idx:
        name_to_idx[group] = len(labels)
        labels.append(group)

for _, row in aggregated.iterrows():
    source.append(name_to_idx[row['pre_group']])
    target.append(name_to_idx[row['post_group']])
    value.append(row['weight'])


name_to_color = { # Key by group name for easier identification
    "Ir94e": {"line": "purple", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T1 (L)": {"line": "yellow", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T1 (R)": {"line": "yellow", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T2 (L)": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "GNG.SLP.T2 (R)": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Earmuff": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Stanley Glu interneuron": {"line": "red", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Stanley ACh interneuron": {"line": "red", "arrow": "rgba(0, 255, 0, 0.5)"},
    "OviDN": {"line": "blue", "arrow": "rgba(0, 255, 0, 0.5)"},
}

node_colors = [name_to_color.get(label, {"line": "grey"})['line'] for label in labels]
arrow_colors = [name_to_color.get(labels[src], {"arrow": "rgba(128, 128, 128, 0.5)"})['arrow'] for src in source]

# plot Sankey
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        color=node_colors
    ),
    link=dict(
        arrowlen=15,
        source=source,
        target=target,
        value=value,
        color=arrow_colors
    )
)])
fig.update_layout(title_text="Ir94e → interneuron groups → OviDN (aggregated)", font_size=10)
fig.show()

## Initial conclusions prior to characterizations of "Other" neurons

Synapses between known groups using v630 matches exactly what we see in the paper, which is great! v783 only adds more snyapses, which is also a good sign.

In both snapshots, Ir94e synapses onto itself (which makes sense biologically). It likely didn't show up in the paper for simplicity.

The big question lies in the "Other" neurons. They weren't mentioned in the paper yet play a big role in the v630 Sankey. They're even more prominent in the v783 circuit. Their characterization will tell us their significance in the function of the circuit.

# Part 3: Characterization of unknown neurons

We'll start with v783.

## 3.1 Identify unknown neurons

In [73]:
import pandas as pd
import json

edge_list = pd.read_csv("filtered_edge_list.csv")
# G = nx.from_pandas_edgelist(edge_list, source="pre_root_id", target="post_root_id", create_using=nx.DiGraph)

with open("data/groups.json") as f:
    groups = json.load(f)["783"]

# invert groups -> id_to_group
id_to_group = {}
for grp, ids in groups.items():
    for i in ids:
        id_to_group[str(i)] = grp

pre_groups = edge_list['pre_root_id'].apply(lambda i: id_to_group.get(str(i), None)).rename('group')
post_groups = edge_list['post_root_id'].apply(lambda i: id_to_group.get(str(i), None)).rename('group')
pre_df = pd.concat([edge_list['pre_root_id'], pre_groups], axis=1).rename(columns={'pre_root_id': 'root_id'})
post_df = pd.concat([edge_list['post_root_id'], post_groups], axis=1).rename(columns={'post_root_id': 'root_id'})

# print(pre_df)
# print(post_df)

unknown_neurons = pd.concat([pre_df, post_df])
unknown_neurons = unknown_neurons[unknown_neurons['group'].isnull()]['root_id'].unique()

## 3.2 Merge onto cell types table

In [74]:
cell_types = pd.read_csv("consolidated_cell_types.csv")
unknown_cell_types = cell_types[cell_types['root_id'].isin(unknown_neurons)]
print(unknown_cell_types)
print(unknown_cell_types.shape)
unknown_cell_types.to_csv("filtered_cell_types.csv", index=False)

                   root_id  primary_type additional_type(s)
167     720575940604891360  AN_GNG_PRW_3                NaN
204     720575940605040300      LHAD1f4b      LHAD1f4, pSP3
869     720575940608471900        CB0627        hb702618710
3138    720575940616404820          mAL4            mAL_fru
7562    720575940626132030      LHAD1f4c      LHAD1f4, pSP3
...                    ...           ...                ...
122991  720575940637956698        SLP234                NaN
131439  720575940625005239        CB0627        hb702618710
132069  720575940626622339  AN_GNG_PRW_4                NaN
133384  720575940629013199        CB0550                NaN
136282  720575940635210859       DNpe007       hb1343489608

[67 rows x 3 columns]
(67, 3)


In [75]:
unknown_ids = set(unknown_cell_types['root_id'])
type_map = unknown_cell_types.set_index('root_id')['primary_type'].to_dict()
nt_map = (edge_list.groupby('pre_root_id')['nt_type']
          .agg(lambda x: sorted(set(x))[0] if len(set(x)) == 1 else "MIXED")
          .to_dict())

out_w = edge_list[edge_list['pre_root_id'].isin(unknown_ids)].groupby('pre_root_id')['syn_count'].sum()
in_w  = edge_list[edge_list['post_root_id'].isin(unknown_ids)].groupby('post_root_id')['syn_count'].sum()

summary = pd.DataFrame({'out_weight': out_w, 'in_weight': in_w}).fillna(0).astype(int)
summary['total_weight'] = summary['out_weight'] + summary['in_weight']
summary['cell_type'] = summary.index.map(type_map)
summary['nt_type'] = summary.index.map(nt_map)
summary.sort_values('total_weight', ascending=False)
summary.to_csv("summary.csv", index=False)

In [76]:
nt_scores = pd.read_csv("neurons.csv")
nt_scores = nt_scores[nt_scores['root_id'].isin(unknown_ids)]
nt_columns = ['da_avg', 'ser_avg', 'gaba_avg', 'glut_avg', 'ach_avg', 'oct_avg']

sorted_scores = nt_scores[nt_columns].apply(lambda r: sorted(r, reverse=True)[:2], axis=1, result_type='expand')
nt_scores['top_score'] = sorted_scores[0]
nt_scores['margin'] = sorted_scores[0] - sorted_scores[1]

# candidates for "genuinely unspecified": low absolute confidence OR small margin
low_confidence = nt_scores[(nt_scores['top_score'] < 0.5) | (nt_scores['margin'] < 0.15)]
low_confidence.to_csv("low_confidence.csv", index=False)
print(low_confidence.head())
print(low_confidence.shape)

                   root_id    group nt_type  nt_type_score  da_avg  ser_avg  \
1056    720575940604891360      GNG     NaN            0.0    0.05     0.04   
6068    720575940608471900      GNG     NaN            0.0    0.08     0.06   
54652   720575940622319715  SMP.FLA     NaN            0.0    0.03     0.02   
59717   720575940623207885      GNG     NaN            0.0    0.05     0.06   
122070  720575940636990064  SLP.SMP     NaN            0.0    0.32     0.30   

        gaba_avg  glut_avg  ach_avg  oct_avg  top_score  margin  
1056        0.45      0.09     0.37      0.0       0.45    0.08  
6068        0.38      0.34     0.13      0.0       0.38    0.04  
54652       0.43      0.46     0.07      0.0       0.46    0.03  
59717       0.39      0.11     0.39      0.0       0.39    0.00  
122070      0.08      0.10     0.20      0.0       0.32    0.02  
(5, 12)


In [77]:
nt_scores = pd.read_csv("neurons.csv")
nt_scores = nt_scores[nt_scores['root_id'].isin(unknown_ids)]
no_nt = nt_scores[nt_scores['nt_type'].isna() | ~(nt_scores['nt_type_score'] > 0)]
no_nt.to_csv("no_nt.csv", index=False)
print(low_confidence['nt_type'])

1056      NaN
6068      NaN
54652     NaN
59717     NaN
122070    NaN
Name: nt_type, dtype: object


In [78]:
edge_list = pd.read_csv("filtered_edge_list.csv")
nt_scores = pd.read_csv('neurons.csv')

with open("data/groups.json") as f:
    groups = json.load(f)["783"]

# invert groups -> id_to_group
id_to_group = {}
for grp, ids in groups.items():
    for i in ids:
        id_to_group[str(i)] = grp

pre_groups = edge_list['pre_root_id'].apply(lambda i: id_to_group.get(str(i), None)).rename('group')
post_groups = edge_list['post_root_id'].apply(lambda i: id_to_group.get(str(i), None)).rename('group')
pre_df = pd.concat([edge_list['pre_root_id'], pre_groups], axis=1).rename(columns={'pre_root_id': 'root_id'})
post_df = pd.concat([edge_list['post_root_id'], post_groups], axis=1).rename(columns={'post_root_id': 'root_id'})

unknown_neurons = pd.concat([pre_df, post_df])
unknown_neurons = unknown_neurons[unknown_neurons['group'].isnull()]['root_id'].unique()

edge_list = edge_list[edge_list['pre_root_id'].isin(unknown_ids)].drop(columns=['post_root_id'])
merged = edge_list.merge(nt_scores, left_on='pre_root_id', right_on='root_id', how='inner').drop(columns=['da_avg', 'ser_avg','gaba_avg', 'glut_avg', 'ach_avg', 'oct_avg', 'group'])
merged = merged[merged['nt_type_x'] != merged['nt_type_y']]
merged.to_csv('disagreements.csv', index=False)

In [79]:
cell_types = pd.read_csv("filtered_cell_types.csv")

cell_types_unconfident = merged.merge(cell_types, on='root_id', how='inner').drop_duplicates(subset=['root_id'])[['root_id', 'primary_type', 'nt_type_x']]
unconfident_ids = set(cell_types_unconfident['root_id'])
series_unconfident_cell_types = cell_types_unconfident['primary_type'].unique()
cell_types_all = cell_types[cell_types['primary_type'].isin(series_unconfident_cell_types)].drop_duplicates()
cell_types_all['low_conf'] = cell_types['root_id'].isin(unconfident_ids)

summary = cell_types_all.groupby('primary_type').agg(n_members=('root_id', 'count'), n_low_conf=('low_conf', 'sum'))
summary['has_confident_neighbor'] = summary['n_members'] > summary['n_low_conf']

print(summary)


              n_members  n_low_conf  has_confident_neighbor
primary_type                                               
AN_GNG_PRW_3          2           2                   False
CB0627                2           1                    True
SLP212c               1           1                   False
SMP286                1           1                   False


In [80]:
edge_list = pd.read_csv("filtered_edge_list.csv")
cell_types = pd.read_csv("filtered_cell_types.csv")
type_map = cell_types.set_index('root_id')['primary_type'].to_dict()

with open("data/groups.json") as f:
    groups = json.load(f)["783"]

id_to_group = {}
for grp, ids in groups.items():
    for i in ids:
        id_to_group[i] = grp

IR94E = set(groups['Ir94e'])
OVIDN = set(groups['OviDN'])

hop1_edges = edge_list[edge_list['pre_root_id'].isin(IR94E) & ~(edge_list['post_root_id'].isin(id_to_group))].copy()
hop1_edges['type'] = hop1_edges['post_root_id'].map(type_map)
unknown_hop1_interneurons = set(hop1_edges['post_root_id'])

all_hop1_interneurons = set(edge_list[edge_list['pre_root_id'].isin(IR94E)]['post_root_id']) - IR94E

hop2_edges = edge_list[
    (edge_list['pre_root_id'].isin(all_hop1_interneurons) & ~(edge_list['post_root_id'].isin(id_to_group)))
].copy()
hop2_edges['type'] = hop2_edges['post_root_id'].map(type_map)
unknown_hop2_interneurons = set(hop2_edges['post_root_id']) - unknown_hop1_interneurons

hop1_named = set(hop1_edges.groupby('type').sum()['syn_count'][lambda s: s >= 100].index)
hop2_named = set(hop2_edges.groupby('type').sum()['syn_count'][lambda s: s >= 100].index)

print("hop1 named:", hop1_named)
print(f"H1 interneurons ({len(unknown_hop1_interneurons)}): {unknown_hop1_interneurons}")
print("hop2 named:", hop2_named)
print(f"H2 interneurons ({len(unknown_hop2_interneurons)}): {unknown_hop2_interneurons}")

hop1 named: {'CB0437', 'SLP234', 'CB0159'}
H1 interneurons (30): {720575940626622339, 720575940620553094, 720575940620445062, 720575940627444490, 720575940642335373, 720575940626941080, 720575940634090777, 720575940622695448, 720575940617539355, 720575940624928284, 720575940614635175, 720575940610956334, 720575940614128691, 720575940628683063, 720575940617299515, 720575940625310014, 720575940637763135, 720575940623207885, 720575940624266061, 720575940616404820, 720575940637956698, 720575940622335197, 720575940604891360, 720575940635642725, 720575940623696362, 720575940635210859, 720575940613524973, 720575940645503854, 720575940611411057, 720575940641284853}
hop2 named: {'VESa2_P01', 'SLP236', 'SMP550', 'CB0550', 'AVLP315'}
H2 interneurons (37): {720575940628771587, 720575940647473796, 720575940620122503, 720575940612708374, 720575940633926295, 720575940622151704, 720575940635807130, 720575940632603548, 720575940632556831, 720575940620038433, 720575940615626018, 720575940624007207, 7205

In [81]:
nt_types = pd.read_csv('neurons.csv')
nt_type_map = nt_types.set_index('root_id')['nt_type'].to_dict()

def bucket_unknown(root_id):
    cell_type = type_map[root_id]
    nt_type = nt_type_map.get(root_id) 

    if root_id in unconfident_ids:
        print(f"{root_id} has unknown NT type")
        return "Unknown"
    
    named_set = None
    if root_id in unknown_hop1_interneurons: named_set = hop1_named
    elif root_id in unknown_hop2_interneurons: named_set = hop2_named

    if cell_type in named_set:
        return cell_type
    
    return f"{"1°" if named_set == hop1_named else "2°"} {nt_type} interneuron"

for u in unknown_neurons:
    if id_to_group.get(u): continue
    id_to_group[u] = bucket_unknown(u)

print(id_to_group)

720575940604891360 has unknown NT type
720575940608471900 has unknown NT type
720575940622319715 has unknown NT type
720575940623207885 has unknown NT type
720575940636990064 has unknown NT type
{720575940621375231: 'Ir94e', 720575940638218173: 'Ir94e', 720575940626016017: 'Ir94e', 720575940631082124: 'Ir94e', 720575940610683315: 'Ir94e', 720575940612920386: 'Ir94e', 720575940614211295: 'Ir94e', 720575940624079544: 'Ir94e', 720575940628198503: 'Ir94e', 720575940627438906: 'Ir94e', 720575940625450498: 'Ir94e', 720575940621898665: 'Ir94e', 720575940627402568: 'Ir94e', 720575940643065032: 'Ir94e', 720575940611849178: 'Ir94e', 720575940637747519: 'Ir94e', 720575940625696601: 'Ir94e', 720575940638813016: 'Ir94e', 720575940632512156: 'OviDN', 720575940620625880: 'OviDN', 720575940621257340: 'OviDN', 720575940627921182: 'OviDN', 720575940642312136: 'OviDN', 720575940629861163: 'Earmuff', 720575940617671285: 'GNG.SLP.T1 (L)', 720575940616759014: 'GNG.SLP.T1 (R)', 720575940619034782: 'GNG.SLP.T

In [82]:
import json
import pandas as pd

edge_list = pd.read_csv("filtered_edge_list.csv")
with open("data/groups.json") as f:
    groups = json.load(f)["783"]

def id_to_grp(i):
    return id_to_group.get(i, "Other")

edge_list['pre_group'] = edge_list['pre_root_id'].apply(id_to_grp)
edge_list['post_group'] = edge_list['post_root_id'].apply(id_to_grp)

edge_list.groupby(['pre_group', 'post_group'])['syn_count'].sum().reset_index().to_csv("grouped_edge_list.csv", index=False)

In [ ]:
import plotly.graph_objects as go

aggregated = pd.read_csv("grouped_edge_list.csv")

# Build global labels
name_to_idx = {}
labels = []

source = []
target = []
value = []

for group in aggregated['pre_group'].unique():
    if group not in name_to_idx:
        name_to_idx[group] = len(labels)
        labels.append(group)

for group in aggregated['post_group'].unique():
    if group not in name_to_idx:
        name_to_idx[group] = len(labels)
        labels.append(group)

for _, row in aggregated.iterrows():
    source.append(name_to_idx[row['pre_group']])
    target.append(name_to_idx[row['post_group']])
    value.append(row['syn_count'])

nt_to_color = {
    "ACH": "rgba(0, 255, 0, 0.5)",
    "GABA": "rgba(255, 0, 0, 0.5)",
    "GLUT": "rgba(255, 0, 0, 0.5)",
}

name_to_color = { # Key by group name for easier identification
    "Ir94e": {"line": "purple", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T1 (L)": {"line": "yellow", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T1 (R)": {"line": "yellow", "arrow": "rgba(0, 255, 0, 0.5)"},
    "GNG.SLP.T2 (L)": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "GNG.SLP.T2 (R)": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Earmuff": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Stanley Glu interneuron": {"line": "red", "arrow": "rgba(255, 0, 0, 0.5)"},
    "Stanley ACh interneuron": {"line": "red", "arrow": "rgba(0, 255, 0, 0.5)"},
    "OviDN": {"line": "blue", "arrow": "rgba(0, 255, 0, 0.5)"},
    "1° ACH interneuron": {"line": "gray", "arrow": "rgba(0, 255, 0, 0.5)"},
    "1° GABA interneuron": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "1° GLUT interneuron": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "1° SER interneuron": {"line": "yellow", "arrow": "rgba(0, 0, 255, 0.5)"},
    "2° ACH interneuron": {"line": "gray", "arrow": "rgba(0, 255, 0, 0.5)"},
    "2° GABA interneuron": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "2° GLUT interneuron": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "AVLP315": {"line": "gray", "arrow": "rgba(0, 255, 0, 0.5)"},
    "CB0159": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "CB0437": {"line": "gray", "arrow": "rgba(0, 255, 0, 0.5)"},
    "CB0550": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"},
    "SLP234": {"line": "gray", "arrow": "rgba(0, 255, 0, 0.5)"},
    "SLP236": {"line": "gray", "arrow": "rgba(0, 255, 0, 0.5)"},
    "SMP550": {"line": "gray", "arrow": "rgba(0, 255, 0, 0.5)"},
    "Unknown": {"line": "gray", "arrow": "rgba(128, 128, 128, 0.5)"},
    "VESa2_P01": {"line": "yellow", "arrow": "rgba(255, 0, 0, 0.5)"}

}

node_colors = [name_to_color.get(label, {"line": "grey"})['line'] for label in labels]
arrow_colors = [name_to_color.get(labels[src], {"arrow": "rgba(128, 128, 128, 0.5)"})['arrow'] for src in source]

# plot Sankey
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        color=node_colors
    ),
    link=dict(
        arrowlen=15,
        source=source,
        target=target,
        value=value,
        color=arrow_colors
    )
)])
fig.update_layout(title_text="Ir94e → interneuron groups → OviDN (aggregated)", font_size=10)
fig.show()

In [84]:
h1_ach = [u for u in unknown_neurons if id_to_group[u] == "1° ACH interneuron"]
unknown = [u for u in unknown_neurons if id_to_group[u] == "Unknown"]

for u in unknown:
    if u in h1_ach: print("overlapping!")